In [1]:
import logging

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CasualSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        assert config.n_embd % config.n_head == 0

        self.n_embd = config.n_embd
        self.n_head = config.n_head

        # key, query, value weight matrix
        self.c_atten: torch.Tensor = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        

    def forward(self, x):
        B, T, C = x.shape

        q, k, v = self.c_atten(x).split(self.n_embd, dim=)  # (B, T, 3C)

class Block(nn.Module):
    """ Transformer Block with self-attention and MLP Layer followed by Layer Normalization with residual connection. """
    def __init__(self, config):
        super().__init__()

        self.ln_1 = nn.LayerNorm(config.n_embd, bias=True)
        self.attn = CasualSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, bias=True)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.module):
    """GPT-2 Implementation"""

    def __init__(self, config):
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.Sequential(*[Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))

        self.lm_head = nn.Linear(config.emd_dim, config.vocab_size, bias=False)

    def forward(self, x):
        B, T = x.shape

        tok_emb = self.transformer.wte(x)  # B, T, C
        pos_emb = self.transformer.wpe(x)  # B, T, C

        x = self.transformer.drop(tok_emb + pos_emb) # B, T, C

        # apply Transformer Block layer
        x = self.transformer.h(x)

        x = self.transformer.ln_f(x)

        x = self.lm_head(x)  # B, T, vocab_size
        

In [50]:
t = torch.arange(90).reshape(2, 3, 15)
t

tensor([[[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14],
         [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29],
         [30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44]],

        [[45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59],
         [60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74],
         [75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89]]])

In [ ]:
t.split(2, dim=0)

(tensor([[[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14],
          [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29],
          [30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44]],
 
         [[45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59],
          [60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74],
          [75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89]]]),)